# RAGAS Evaluation for RAG Pipeline
### PDF → Chunking → FAISS → Query → Answer → Evaluate with RAGAS

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf ragas python-dotenv sacrebleu rouge_score -q

In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
from dotenv import load_dotenv
load_dotenv("E:/ACADEMY CLASS NOTEBOOKS/.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas.metrics import (
    Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall,
    AnswerCorrectness, AnswerSimilarity, ContextRelevance, ContextEntityRecall,
    SemanticSimilarity, AspectCritic, NoiseSensitivity,
    BleuScore, RougeScore, NonLLMStringSimilarity,
)

## Step 1: Load PDF

In [3]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")

Total Pages: 30


## Step 2: Split into Chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 136


## Step 3: Create Embeddings & Store in FAISS

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Retriever & LLM

In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Step 5: Define Test Questions & Ground Truth

In [7]:
test_questions = [
    "Where was Abdul Kalam born?",
    "What year was Abdul Kalam born?",
    "Which institution did Kalam study aerospace engineering at?",
    "What was Kalam's role at DRDO from 1992 to 1999?",
    "What was the name of the coronary stent Kalam co-developed?",
]

ground_truths = [
    "Abdul Kalam was born in Rameswaram, Tamil Nadu.",
    "Abdul Kalam was born in 1931.",
    "Kalam studied aerospace engineering at the Madras Institute of Technology (MIT).",
    "Kalam served as the Director General of DRDO from 1992 to 1999.",
    "Kalam co-developed a low-cost coronary stent called the 'Kalam-Raju Stent'.",
]

print(f"Total Test Questions: {len(test_questions)}")

Total Test Questions: 5


## Step 6: Generate Answers from RAG Pipeline

In [8]:
answers = []
contexts = []

for query in test_questions:
    retrieved_docs = retriever.invoke(query)
    context = [doc.page_content for doc in retrieved_docs]
    contexts.append(context)

    context_str = "\n\n".join(context)
    prompt = f"""Answer the question based only on the following context:

{context_str}

Question: {query}
Answer:"""

    response = llm.invoke(prompt)
    answers.append(response.content)
    print(f"Q: {query}")
    print(f"A: {response.content}\n")

Q: Where was Abdul Kalam born?
A: Abdul Kalam was born in Rameswaram, Tamil Nadu.

Q: What year was Abdul Kalam born?
A: Abdul Kalam was born on 15 October 1931.

Q: Which institution did Kalam study aerospace engineering at?
A: Kalam studied aerospace engineering at the Madras Institute of Technology.

Q: What was Kalam's role at DRDO from 1992 to 1999?
A: Kalam served as the chief scientific adviser to the prime minister and secretary of the Defence Research and Development Organisation (DRDO) from July 1992 to December 1999.

Q: What was the name of the coronary stent Kalam co-developed?
A: The coronary stent co-developed by Kalam was named the "Kalam-Raju stent."



In [9]:

print("=" * 80)
print("ANSWER vs GROUND TRUTH COMPARISON")
print("=" * 80)

for i in range(len(test_questions)):
    print(f"\n📌 Q{i+1}: {test_questions[i]}")
    print(f"\n  ✅ Ground Truth  : {ground_truths[i]}")
    print(f"  🤖 RAG Answer    : {answers[i]}")
    print("-" * 80)


ANSWER vs GROUND TRUTH COMPARISON

📌 Q1: Where was Abdul Kalam born?

  ✅ Ground Truth  : Abdul Kalam was born in Rameswaram, Tamil Nadu.
  🤖 RAG Answer    : Abdul Kalam was born in Rameswaram, Tamil Nadu.
--------------------------------------------------------------------------------

📌 Q2: What year was Abdul Kalam born?

  ✅ Ground Truth  : Abdul Kalam was born in 1931.
  🤖 RAG Answer    : Abdul Kalam was born on 15 October 1931.
--------------------------------------------------------------------------------

📌 Q3: Which institution did Kalam study aerospace engineering at?

  ✅ Ground Truth  : Kalam studied aerospace engineering at the Madras Institute of Technology (MIT).
  🤖 RAG Answer    : Kalam studied aerospace engineering at the Madras Institute of Technology.
--------------------------------------------------------------------------------

📌 Q4: What was Kalam's role at DRDO from 1992 to 1999?

  ✅ Ground Truth  : Kalam served as the Director General of DRDO from 1992 to 1

## Step 7: Prepare RAGAS Evaluation Dataset

In [10]:
samples = []

for i in range(len(test_questions)):
    sample = SingleTurnSample(
        user_input=test_questions[i],
        response=answers[i],
        retrieved_contexts=contexts[i],
        reference=ground_truths[i],
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)

print(f"Evaluation Dataset Created with {len(eval_dataset)} samples")

Evaluation Dataset Created with 5 samples


## Step 8: Setup Evaluator

In [11]:
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
run_config = RunConfig(timeout=120, max_retries=2, max_workers=4)

print("Evaluator Ready (GPT-4o-mini + OpenAI Embeddings)")

Evaluator Ready (GPT-4o-mini + OpenAI Embeddings)


---
# Generation Metrics (Answer Quality)
---

### 1. Faithfulness
Is the answer grounded in the retrieved context? (0 → 1)

In [12]:
result = evaluate(dataset=eval_dataset, metrics=[Faithfulness(llm=evaluator_llm)], run_config=run_config)
result.to_pandas()[["user_input", "faithfulness"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,faithfulness
0,Where was Abdul Kalam born?,1.0
1,What year was Abdul Kalam born?,1.0
2,Which institution did Kalam study aerospace en...,1.0
3,What was Kalam's role at DRDO from 1992 to 1999?,1.0
4,What was the name of the coronary stent Kalam ...,1.0


### 2. Response Relevancy
Is the answer relevant to the question? (0 → 1)

In [13]:
result = evaluate(dataset=eval_dataset, metrics=[ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)], run_config=run_config)
result.to_pandas()[["user_input", "answer_relevancy"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,answer_relevancy
0,Where was Abdul Kalam born?,1.000000
1,What year was Abdul Kalam born?,0.975204
2,Which institution did Kalam study aerospace en...,0.730419
3,What was Kalam's role at DRDO from 1992 to 1999?,0.751542
4,What was the name of the coronary stent Kalam ...,0.972119


### 3. Answer Correctness
Factual correctness of the answer compared to ground truth (0 → 1)

In [14]:
result = evaluate(dataset=eval_dataset, metrics=[AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings)], run_config=run_config)
result.to_pandas()[["user_input", "answer_correctness"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,answer_correctness
0,Where was Abdul Kalam born?,1.000000
1,What year was Abdul Kalam born?,0.970528
2,Which institution did Kalam study aerospace en...,0.991848
3,What was Kalam's role at DRDO from 1992 to 1999?,0.214200
4,What was the name of the coronary stent Kalam ...,0.597605


### 4. Answer Similarity
Semantic similarity between answer and ground truth (0 → 1)

In [16]:
result = evaluate(dataset=eval_dataset, metrics=[AnswerSimilarity(embeddings=evaluator_embeddings)], run_config=run_config)
result.to_pandas()[["user_input", "answer_similarity"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,answer_similarity
0,Where was Abdul Kalam born?,1.000000
1,What year was Abdul Kalam born?,0.882113
2,Which institution did Kalam study aerospace en...,0.967391
3,What was Kalam's role at DRDO from 1992 to 1999?,0.856799
4,What was the name of the coronary stent Kalam ...,0.890419


---
# Retrieval Metrics (Context Quality)
---

### 5. Context Precision
Are the relevant chunks ranked higher in the retrieved results? (0 → 1)

In [17]:
result = evaluate(dataset=eval_dataset, metrics=[LLMContextPrecisionWithoutReference(llm=evaluator_llm)], run_config=run_config)
result.to_pandas()[["user_input", "llm_context_precision_without_reference"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,llm_context_precision_without_reference
0,Where was Abdul Kalam born?,1.0
1,What year was Abdul Kalam born?,1.0
2,Which institution did Kalam study aerospace en...,1.0
3,What was Kalam's role at DRDO from 1992 to 1999?,1.0
4,What was the name of the coronary stent Kalam ...,1.0


### 6. Context Recall
Does the retrieved context cover the ground truth? (0 → 1)

In [18]:
result = evaluate(dataset=eval_dataset, metrics=[LLMContextRecall(llm=evaluator_llm)], run_config=run_config)
result.to_pandas()[["user_input", "context_recall"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,context_recall
0,Where was Abdul Kalam born?,1.0
1,What year was Abdul Kalam born?,1.0
2,Which institution did Kalam study aerospace en...,1.0
3,What was Kalam's role at DRDO from 1992 to 1999?,0.0
4,What was the name of the coronary stent Kalam ...,1.0


### 7. Context Relevance
How relevant is the retrieved context to the question? (0 → 1)

In [19]:
result = evaluate(dataset=eval_dataset, metrics=[ContextRelevance(llm=evaluator_llm)], run_config=run_config)
result.to_pandas()[["user_input", "nv_context_relevance"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,nv_context_relevance
0,Where was Abdul Kalam born?,1.0
1,What year was Abdul Kalam born?,1.0
2,Which institution did Kalam study aerospace en...,1.0
3,What was Kalam's role at DRDO from 1992 to 1999?,1.0
4,What was the name of the coronary stent Kalam ...,1.0


### 8. Context Entity Recall
Does the retrieved context capture the key entities from the ground truth? (0 → 1)

In [20]:
result = evaluate(dataset=eval_dataset, metrics=[ContextEntityRecall(llm=evaluator_llm)], run_config=run_config)
result.to_pandas()[["user_input", "context_entity_recall"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,context_entity_recall
0,Where was Abdul Kalam born?,0.666667
1,What year was Abdul Kalam born?,0.000000
2,Which institution did Kalam study aerospace en...,0.250000
3,What was Kalam's role at DRDO from 1992 to 1999?,0.400000
4,What was the name of the coronary stent Kalam ...,0.000000


---
# End-to-End Metrics
---

### 9. Semantic Similarity
Embedding-based similarity between answer and ground truth (0 → 1)

In [21]:
result = evaluate(dataset=eval_dataset, metrics=[SemanticSimilarity(embeddings=evaluator_embeddings)], run_config=run_config)
result.to_pandas()[["user_input", "semantic_similarity"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,semantic_similarity
0,Where was Abdul Kalam born?,1.000000
1,What year was Abdul Kalam born?,0.882113
2,Which institution did Kalam study aerospace en...,0.967417
3,What was Kalam's role at DRDO from 1992 to 1999?,0.856799
4,What was the name of the coronary stent Kalam ...,0.890419


### 10. Aspect Critic
Custom rubric-based evaluation — LLM judges if answer meets a defined criteria (0 or 1)

In [22]:
result = evaluate(
    dataset=eval_dataset,
    metrics=[AspectCritic(
        name="correctness",
        definition="Is the answer factually correct and matches the ground truth?",
        llm=evaluator_llm,
    )],
    run_config=run_config,
)
result.to_pandas()[["user_input", "correctness"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,correctness
0,Where was Abdul Kalam born?,1
1,What year was Abdul Kalam born?,1
2,Which institution did Kalam study aerospace en...,1
3,What was Kalam's role at DRDO from 1992 to 1999?,0
4,What was the name of the coronary stent Kalam ...,1


### 11. Noise Sensitivity
How sensitive is the answer to irrelevant/noisy context? (0 → 1, lower is better)

In [23]:
result = evaluate(dataset=eval_dataset, metrics=[NoiseSensitivity(llm=evaluator_llm)], run_config=run_config)
df = result.to_pandas()
noise_cols = [c for c in df.columns if "noise" in c.lower()]
print("Noise columns found:", noise_cols)
df[["user_input"] + noise_cols]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Noise columns found: ['noise_sensitivity(mode=relevant)']


,user_input,noise_sensitivity(mode=relevant)
0,Where was Abdul Kalam born?,0.0
1,What year was Abdul Kalam born?,1.0
2,Which institution did Kalam study aerospace en...,0.0
3,What was Kalam's role at DRDO from 1992 to 1999?,0.0
4,What was the name of the coronary stent Kalam ...,0.0


---
# Additional Metrics (Non-LLM)
---

### 12. BLEU Score
N-gram overlap between answer and ground truth (0 → 1)

In [24]:
result = evaluate(dataset=eval_dataset, metrics=[BleuScore()], run_config=run_config)
result.to_pandas()[["user_input", "bleu_score"]]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,bleu_score
0,Where was Abdul Kalam born?,1.000000
1,What year was Abdul Kalam born?,0.354948
2,Which institution did Kalam study aerospace en...,0.696355
3,What was Kalam's role at DRDO from 1992 to 1999?,0.113720
4,What was the name of the coronary stent Kalam ...,0.074749


### 13. ROUGE Score
Longest common subsequence overlap between answer and ground truth (0 → 1)

In [25]:
result = evaluate(dataset=eval_dataset, metrics=[RougeScore()], run_config=run_config)
df = result.to_pandas()
metric_cols = [c for c in df.columns if c != "user_input"]
df[["user_input"] + metric_cols]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,rouge_score(mode=fmeasure)
0,Where was Abdul Kalam born?,[Organisation\nWebsite A. P. J. Abdul Kalam Ce...,"Abdul Kalam was born in Rameswaram, Tamil Nadu.","Abdul Kalam was born in Rameswaram, Tamil Nadu.",1.000000
1,What year was Abdul Kalam born?,[Organisation\nWebsite A. P. J. Abdul Kalam Ce...,Abdul Kalam was born on 15 October 1931.,Abdul Kalam was born in 1931.,0.714286
2,Which institution did Kalam study aerospace en...,"[This was my first stage, in which I learnt\nl...",Kalam studied aerospace engineering at the Mad...,Kalam studied aerospace engineering at the Mad...,0.952381
3,What was Kalam's role at DRDO from 1992 to 1999?,[US$780 million in 2023) for the project title...,Kalam served as the chief scientific adviser t...,Kalam served as the Director General of DRDO f...,0.512821
4,What was the name of the coronary stent Kalam ...,[nuclear tests conducted in May 1998.[33] Alon...,The coronary stent co-developed by Kalam was n...,Kalam co-developed a low-cost coronary stent c...,0.461538


### 14. String Similarity (Non-LLM)
Levenshtein distance-based string similarity between answer and ground truth (0 → 1)

In [26]:
result = evaluate(dataset=eval_dataset, metrics=[NonLLMStringSimilarity()], run_config=run_config)
df = result.to_pandas()
metric_cols = [c for c in df.columns if c != "user_input"]
df[["user_input"] + metric_cols]

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,non_llm_string_similarity
0,Where was Abdul Kalam born?,[Organisation\nWebsite A. P. J. Abdul Kalam Ce...,"Abdul Kalam was born in Rameswaram, Tamil Nadu.","Abdul Kalam was born in Rameswaram, Tamil Nadu.",1.00000
1,What year was Abdul Kalam born?,[Organisation\nWebsite A. P. J. Abdul Kalam Ce...,Abdul Kalam was born on 15 October 1931.,Abdul Kalam was born in 1931.,0.70000
2,Which institution did Kalam study aerospace en...,"[This was my first stage, in which I learnt\nl...",Kalam studied aerospace engineering at the Mad...,Kalam studied aerospace engineering at the Mad...,0.92500
3,What was Kalam's role at DRDO from 1992 to 1999?,[US$780 million in 2023) for the project title...,Kalam served as the chief scientific adviser t...,Kalam served as the Director General of DRDO f...,0.34104
4,What was the name of the coronary stent Kalam ...,[nuclear tests conducted in May 1998.[33] Alon...,The coronary stent co-developed by Kalam was n...,Kalam co-developed a low-cost coronary stent c...,0.40000
